# Attention Is All You Need

## Learning Objectives

1. Understand scaled dot-product attention and why scaling by √d_k matters
2. Implement multi-head attention from scratch
3. Build a complete Transformer encoder with positional encoding
4. Train a sequence-to-sequence model using Transformers
5. Visualize attention weights and understand what the model learns

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import math

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Level 1: Scaled Dot-Product Attention

Core mechanism: Attention(Q,K,V) = softmax(QK^T/√d_k)V

Why scale by √d_k? Without scaling, softmax becomes too sharp (gradients vanish).
With scaling, softmax stays in a reasonable range for training.

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Scaled dot-product attention: core building block of Transformers"""
    
    def __init__(self, d_k, dropout=0.1):
        super().__init__()
        self.d_k = d_k
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        # Compute attention scores with scaling
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Softmax to get attention weights
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        
        # Multiply by values
        output = torch.matmul(weights, value)
        
        return output, weights


# Test on synthetic data
batch_size = 2
seq_len = 4
d_k = d_v = 64

Q = torch.randn(batch_size, seq_len, d_k, device=device)
K = torch.randn(batch_size, seq_len, d_k, device=device)
V = torch.randn(batch_size, seq_len, d_v, device=device)

attention = ScaledDotProductAttention(d_k, dropout=0.1).to(device)
output, weights = attention(Q, K, V)

print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Weights sum to 1: {weights.sum(dim=-1)[0]}")
print(f"\nAttention matrix:\n{weights[0, 0].detach().cpu().numpy().round(3)}")

## Level 2: Multi-Head Attention

Run h attention heads in parallel, each with different learned projections.
Benefits: ensemble effect, multiple relationship types, same total computation.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention: h attention heads in parallel"""
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention(self.d_k, dropout)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        seq_len = query.size(1)
        
        # Project and reshape
        Q = self.W_q(query).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        attn_output, attn_weights = self.attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.d_model)
        
        # Final projection
        output = self.W_o(attn_output)
        
        return output, attn_weights


# Test multi-head attention
d_model = 512
num_heads = 8
X = torch.randn(batch_size, seq_len, d_model, device=device)

mha = MultiHeadAttention(d_model, num_heads).to(device)
output, attn_weights = mha(X, X, X)

print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Params: {sum(p.numel() for p in mha.parameters()):,}")

## Real-World Example 1: Positional Encoding

Transformers have no recurrence, so we need positional encodings to capture sequence order.
Sinusoidal encodings: each position gets a unique frequency-based vector.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding"""
    
    def __init__(self, d_model, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * 
                             -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:x.size(1)].unsqueeze(0)
        return self.dropout(x)


# Visualize positional encodings
pos_enc = PositionalEncoding(d_model=128, max_seq_len=50)
pe_matrix = pos_enc.pe[:50, :].cpu().detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im = axes[0].imshow(pe_matrix, cmap='viridis', aspect='auto')
axes[0].set_title('Positional Encodings (128D, 50 positions)')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Position')
plt.colorbar(im, ax=axes[0])

for pos in [0, 10, 25, 49]:
    axes[1].plot(pe_matrix[pos, :50], label=f'pos {pos}')
axes[1].set_title('Positional Encoding Values (first 50 dims)')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Encoding Value')
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

print(f"Positional encoding shape: {pe_matrix.shape}")
print(f"Position 0: {pe_matrix[0, :10].round(3)}")
print(f"Position 10: {pe_matrix[10, :10].round(3)}")

## Real-World Example 2: Transformer Block

Complete encoder block: MultiHeadAttention → Residual → LayerNorm → FFN → Residual → LayerNorm

In [ ]:
class FeedForwardNetwork(nn.Module):
    """Position-wise Feed-Forward Network"""
    
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class TransformerBlock(nn.Module):
    """Single Transformer encoder block"""
    
    def __init__(self, d_model, num_heads, d_ff=2048, dropout=0.1):
        super().__init__()
        
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention with residual
        attn_output, _ = self.self_attention(x, x, x, mask)
        x = x + self.dropout(attn_output)
        x = self.norm1(x)
        
        # Feed-forward with residual
        ffn_output = self.ffn(x)
        x = x + self.dropout(ffn_output)
        x = self.norm2(x)
        
        return x


class TransformerEncoder(nn.Module):
    """Full Transformer encoder"""
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len=512):
        super().__init__()
        
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len)
        
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        for layer in self.layers:
            x = layer(x)
        
        x = self.norm(x)
        return x


# Test encoder
vocab_size = 1000
d_model = 256
num_heads = 4
d_ff = 1024
num_layers = 2

encoder = TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers).to(device)

batch_size = 4
seq_len = 16
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
output = encoder(input_ids)

print(f"Input shape: {input_ids.shape}")
print(f"Output shape: {output.shape}")
print(f"Total parameters: {sum(p.numel() for p in encoder.parameters()) / 1e6:.2f}M")

## Real-World Example 3: Sequence Classification

Complete pipeline: Transformer encoder → pooling → classification head
Train on synthetic task to demonstrate end-to-end training.

In [ ]:
class TransformerClassifier(nn.Module):
    """Transformer-based sequence classifier"""
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, num_classes):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers)
        self.classifier = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        encoded = self.encoder(x)
        pooled = encoded.mean(dim=1)
        logits = self.classifier(pooled)
        return logits


# Model and training setup
model = TransformerClassifier(
    vocab_size=1000, d_model=128, num_heads=4, d_ff=512, 
    num_layers=2, num_classes=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Synthetic dataset
n_samples = 100
X = torch.randint(0, 1000, (n_samples, 16))
y = torch.randint(0, 2, (n_samples,))

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Training loop
num_epochs = 10
train_losses = []
accuracies = []

for epoch in range(num_epochs):
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item() * batch_X.size(0)
        predictions = logits.argmax(dim=1)
        total_correct += (predictions == batch_y).sum().item()
        total_samples += batch_X.size(0)
    
    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    train_losses.append(epoch_loss)
    accuracies.append(epoch_acc)
    
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}: Loss={epoch_loss:.4f}, Acc={epoch_acc:.3f}")

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid()

axes[1].plot(accuracies)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].grid()

plt.tight_layout()
plt.show()

print(f"\nFinal Loss: {train_losses[-1]:.4f}")
print(f"Final Accuracy: {accuracies[-1]:.3f}")

## Key Takeaways

**Core mechanism:** Self-attention = softmax(QK^T/√d_k)V
- Query-key dot product measures relevance
- Scaling by √d_k stabilizes gradients
- Softmax produces attention weights (0-1)
- Applied to values for output

**Multi-head attention:** h independent attention heads
- Different heads learn different relationships
- Ensemble effect improves robustness
- Same total computation as single head

**Positional encoding:** Sinusoidal embeddings
- Captures absolute/relative position
- Generalizes to longer sequences
- Enables position-aware self-attention

**Transformer block:** Attention + FFN + Residuals + LayerNorm
- Stack 6-12+ blocks for depth
- Residual connections enable deep networks
- Pre-norm architecture more stable

**Why it works:**
1. Parallelizable: all positions in parallel (vs RNN sequential)
2. Long-range: O(N) path between any two positions (vs RNN O(N) but slow)
3. Interpretable: attention weights show what matters
4. Scalable: foundation of billion-parameter models

**Related papers:**
- [BERT](./02-bert.md) - Bidirectional pre-training
- [GPT-3](./03-gpt3.md) - Autoregressive scaling
- [Vision Transformer](../vision/concepts/02-vision-transformer.md) - Applied to images